# Candle features and next-bar target table
Research-only notebook. No model training or database writes. Quantile cutoffs and K-means are fitted on the training period only.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text
from stock_ml_dynamic.db import engine
from stock_ml_dynamic.candle_table import (build_candle_table, apply_training_transformers, NUMERIC_FEATURES, CATEGORICAL_FEATURES, LABEL_MAPPING)

## 1. Select market data
Use a single symbol and interval. Rows are ordered by timestamp. Adjusted close is taken from `adj_close_price`; rows with missing adjusted close are not silently substituted.

In [ ]:
SYMBOL = "AAPL"
INTERVAL = "1d"
LIMIT = 2000
TRAIN_FRACTION = 0.75
ZSCORE_WINDOW = 30
SELECTED_K = None  # e.g. 3 to override the elbow heuristic
K_VALUES = range(2, 9)
TIMEZONE = "America/New_York"

In [ ]:
query = text("""SELECT symbol, interval_code, bar_timestamp, open_price, high_price, low_price, close_price, adj_close_price, volume
FROM (SELECT symbol, interval_code, bar_timestamp, open_price, high_price, low_price, close_price, adj_close_price, volume
      FROM raw.stock_prices WHERE symbol=:symbol AND interval_code=:interval
      ORDER BY bar_timestamp DESC LIMIT :limit) AS recent
ORDER BY bar_timestamp""")
with engine().connect() as conn:
    prices = pd.read_sql(query, conn, params={"symbol": SYMBOL, "interval": INTERVAL, "limit": LIMIT})
print(f"Loaded {len(prices):,} bars for {SYMBOL} {INTERVAL}")
display(prices.tail())

## 2. Compute causal features
Candle percentages, gap, trailing z-score (previous 30 closes only), three-point quadratic derivative and calendar fields. The final row has no known next-bar target.

In [ ]:
frame = build_candle_table(prices, z_window=ZSCORE_WINDOW, timestamp_timezone=TIMEZONE)
train_end = int(len(frame)*TRAIN_FRACTION)
print("Training boundary:", frame.iloc[train_end]["bar_timestamp"])
display(frame[["bar_timestamp", "adj_close"] + NUMERIC_FEATURES + ["next_return"]].tail(10))

## 3. Training-fitted K-means elbow and quantile targets
Only the training period determines cluster centers and target cutoffs. Elbow selection is heuristic; inspect the chart and set `SELECTED_K` above to override.

In [ ]:
table, quantiles, clusters = apply_training_transformers(frame, train_end, k_values=K_VALUES, selected_k=SELECTED_K)
print("Chosen K:", clusters.chosen_k_)
print("Training return quantile boundaries:", quantiles.edges_)
display(clusters.inertia_)
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(clusters.inertia_["k"], clusters.inertia_["inertia"], marker="o")
ax.axvline(clusters.chosen_k_, linestyle="--", label="selected k")
ax.set(xlabel="Number of clusters", ylabel="K-means inertia", title="Training-only elbow analysis")
ax.legend(); plt.show()

## 4. Review the complete feature/target table
`direction`: 2 = down, 0 = no_change, 1 = up. `adj_close` is the current price, **not** a future regression label. `next_adj_close` is a one-bar-ahead regression target. Do not include `direction`, `next_return`, or `next_adj_close` in X.

In [ ]:
columns = ["symbol", "interval_code", "bar_timestamp"] + NUMERIC_FEATURES + CATEGORICAL_FEATURES + ["direction", "adj_close", "next_adj_close"]
model_table = table[columns].copy()
display(model_table.tail(12))
print("Rows:", len(model_table), "Train:", train_end, "Post-train:", len(model_table)-train_end)
print("Class mapping:", LABEL_MAPPING)
print("Labeled class distribution:")
display(model_table["direction"].value_counts(dropna=False).sort_index().rename(index=LABEL_MAPPING))

## 5. Validate and inspect
The latest row must have no next-bar target. First 30 rows lack enough trailing history for `adj_z`. Invalid price candles should be investigated before training.

In [ ]:
assert pd.isna(model_table["direction"].iloc[-1])
assert pd.isna(model_table["next_adj_close"].iloc[-1])
assert model_table["adj_z"].iloc[:ZSCORE_WINDOW].isna().all()
parts_sum = model_table[["top_pct","body_pct","bottom_pct"]].sum(axis=1)
valid_range = (prices["high_price"] > prices["low_price"]).to_numpy()
print("Candle-part sum deviations:", int(((parts_sum-1).abs()>1e-6)[valid_range].sum()))
print("Missing values by column:")
display(model_table.isna().sum().to_frame("missing"))

## 6. Prepare modeling matrices (without fitting a classifier)
The split is chronological. For a one-step target, remove the boundary-crossing training row and the unlabeled final row. Fit any one-hot encoder or scaler on training rows only. For regression, use `next_adj_close`, not current `adj_close`.

In [ ]:
feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
train_rows = model_table.iloc[:train_end-1].dropna(subset=feature_cols+["direction","next_adj_close"])
test_rows = model_table.iloc[train_end:-1].dropna(subset=feature_cols+["direction","next_adj_close"])
X_train, X_test = train_rows[feature_cols], test_rows[feature_cols]
y_train_cls, y_test_cls = train_rows["direction"].astype(int), test_rows["direction"].astype(int)
y_train_reg, y_test_reg = train_rows["next_adj_close"], test_rows["next_adj_close"]
print("X train/test:", X_train.shape, X_test.shape)
print("Classification labels:", sorted(y_train_cls.unique().tolist()))
print("Regression targets:", y_train_reg.name, y_test_reg.name)

## 7. Optional local export
This writes a CSV under `artifacts/tables/` only when explicitly enabled. The PostgreSQL backend remains unchanged. For production ML, persist the fitted quantile and K-means objects alongside the model, and never re-fit them on live data.

In [ ]:
EXPORT_CSV = False
if EXPORT_CSV:
    output = ROOT / "artifacts" / "tables" / f"{SYMBOL}_{INTERVAL}_candle_features.csv"
    output.parent.mkdir(parents=True, exist_ok=True)
    model_table.to_csv(output, index=False)
    print("Saved:", output)